In [ ]:
import numpy as np
import pandas as pd
import random
import time
from rapidfuzz import process, fuzz, distance
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import spoa 

# --- 2. DATA GENERATION & UTILS ---
def mutate_sequence(seq, error_rate=0.10):
    if error_rate <= 0: return seq
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            r = random.random()
            if r < 0.5: new_seq.append(random.choice("ACGT")) 
            elif r < 0.75: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT"))
            else: pass 
        else:
            new_seq.append(base)
    return "".join(new_seq)

def gen_dna(k): return "".join(random.choices("ACGT", k=k))

# (Generating fresh data to ensure variables exist for the loop below)
n_items = 100_000 
pool_bc = [gen_dna(12) for _ in range(n_items)]
pool_ins = [gen_dna(50) for _ in range(n_items)]
n_repeat_bc = 500
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(50) for _ in range(n_repeat_bc)]
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(50) for _ in range(n_repeat_bc)]

data = []
for i in range(len(pool_bc)):
     n_reads = random.randint(1, 6) 
     for _ in range(n_reads):
         data.append({
             "ID": i,
             "Barcode": mutate_sequence(pool_bc[i], 0.05), 
             "Insert": mutate_sequence(pool_ins[i], 0.05)
         })

df = pd.DataFrame(data)

barcode_counts = df['Barcode'].value_counts()
all_barcodes = np.array(barcode_counts.index.tolist())

In [ ]:
def encode_msa(msa_strings, alphabet="ACGTN-"):
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    max_len = max(len(s) for s in msa_strings)
    msa_padded = [s.ljust(max_len, '-') for s in msa_strings]
    msa_array = np.array([list(seq) for seq in msa_padded])
    N, L = msa_array.shape
    msa_int = np.zeros((N, L), dtype=np.int8)
    for char, idx in char_to_int.items():
        msa_int[msa_array == char] = idx
    return msa_int, len(alphabet)


def get_marginals(msa_int, vocab_size):
    # Create on hot matrix
    one_hot = np.eye(vocab_size)[msa_int]
    # Calculate frequencies of each base/N/- at each position in MSA
    P_i = one_hot.mean(axis=0)
    return one_hot, P_i

def compute_mi_scores(one_hot, P_i):
    """
    Compute mutual information scores in the MSA
    """
    N, L, A = one_hot.shape
    flat_view = one_hot.transpose(1, 2, 0).reshape(L * A, N)
    joint_probs_flat = (flat_view @ flat_view.T) / N
    P_ij = joint_probs_flat.reshape(L, A, L, A)
    P_product = P_i[:, :, None, None] * P_i[None, None, :, :]
    mask = P_ij > 0
    mi_matrix = np.zeros_like(P_ij)
    mi_matrix[mask] = P_ij[mask] * np.log(P_ij[mask] / P_product[mask])
    return mi_matrix.sum(axis=(1, 3))


def get_elbow_columns(mi_matrix, exclusion_distance=5, verbose=0):
    L = mi_matrix.shape[0]
    mask = np.triu(np.ones((L, L), dtype=bool), k=exclusion_distance + 1)
    rows, cols = np.where(mask)
    scores = mi_matrix[rows, cols]
    
    if len(scores) == 0: return np.array([])

    # 1. Sort Descending
    sorted_indices = np.argsort(scores)[::-1]
    sorted_scores = scores[sorted_indices]
    
    # 2. Determine "Signal End" (Noise Floor Truncation)
    # We stop analyzing where the score drops below the bottom 25% percentile
    noise_floor = np.percentile(scores, 25)
    
    # Keep points strictly ABOVE noise floor for the shape analysis
    # (But keep at least 5 points to allow for geometry, unless total is small)
    valid_mask = sorted_scores > noise_floor
    n_signal_points = np.sum(valid_mask)
    
    # Safety: if signal is super short, take at least top 10% or min 5
    min_points = min(len(sorted_scores), 5)
    cutoff_idx = max(n_signal_points, min_points)
    
    # 3. Truncate for Geometry Calculation
    # We only look for the elbow within the "Signal" region
    curve_y = sorted_scores[:cutoff_idx]
    n_points = len(curve_y)
    
    if n_points < 3:
        # Fallback for tiny signals
        return np.unique(np.concatenate([rows[sorted_indices[:n_points]], cols[sorted_indices[:n_points]]]))

    # 4. Standard Kneedle on Truncated Curve
    x_norm = np.linspace(0, 1, n_points)
    y_norm = (curve_y - curve_y.min()) / (curve_y.max() - curve_y.min() + 1e-9)
    
    # Line from First Point (0, 1) to Last Signal Point (1, 0)
    line_vec = np.array([1.0, -1.0]) # Direction vector (x=1-0, y=0-1)
    line_vec = line_vec / np.linalg.norm(line_vec)
    
    # Vectors from start (0, 1) to all points
    vec_from_start = np.stack([x_norm, y_norm - 1.0], axis=1)
    
    # Cross product (2D) to find distance
    distances = np.abs(vec_from_start[:, 0] * line_vec[1] - vec_from_start[:, 1] * line_vec[0])
    
    # 5. Select
    elbow_idx = np.argmax(distances)
    
    # If the plateau is perfectly flat, elbow_idx might be 0. 
    # In that case, we likely want the whole plateau, not just index 0.
    # We check if the point at elbow_idx is significantly higher than end.
    if elbow_idx == 0:
        # Heuristic: If index 0 is picked, but index 1 is very close in value, walk forward
        # This handles the "perfect plateau" case
        for i in range(1, n_points - 1):
            if curve_y[i] >= (curve_y[0] * 0.95): # Still within 5% of max
                elbow_idx = i
            else:
                break

    n_selected = elbow_idx + 1 # +1 because index is 0-based
    
    # --- Debug Plot (Only if Verbose) ---
    if verbose >= 3:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 3))
        plt.plot(range(n_points), curve_y, '-o', markersize=3, label='Signal Curve')
        plt.axvline(elbow_idx, color='r', linestyle='--', label=f'Elbow (k={n_selected})')
        plt.axhline(noise_floor, color='k', linestyle=':', alpha=0.5, label='Noise Floor')
        plt.title(f"Elbow Analysis (Selecting {n_selected} cols)")
        plt.legend()
        plt.show()

    selected_indices = sorted_indices[:n_selected]

    selected_columns = np.unique(np.concatenate([rows[selected_indices], cols[selected_indices]]))
    
    if verbose >= 3:
        print(f"selected MSA columns: {selected_columns}")
    return selected_columns


def calculate_msa_mi_and_top_cols(barcodes, inserts, verbose=0, mi_exclusion_distance=3):
    barcodes = sub_df['Barcode'].tolist()
    _, msa_barcodes = spoa.poa(barcodes, algorithm=2) 
    inserts = sub_df['Insert'].tolist()
    _, msa_inserts = spoa.poa(inserts, algorithm=2)
    msa_strings = [b + "--------" + i for b, i in zip(msa_barcodes, msa_inserts)]

    # Encode MSA into a numpy array with numbers
    msa_int, vocab_size = encode_msa(msa_strings)

    # Create one hot matrix and the frequencies of each base at each position in MSA
    one_hot, P_i = get_marginals(msa_int, vocab_size)
    
    # Calculate mutual information scores in MSA
    mi_matrix = compute_mi_scores(one_hot, P_i)

    # Use elbow approach to find pairs in MSA with high mutual information
    # Exclude very close positions as sequencing errors of more than one base may 
    # produce spurious mutual info
    top_cols = get_elbow_columns(mi_matrix, mi_exclusion_distance, verbose=verbose)    

    return mi_matrix, top_cols
    

def iterative_hamming_driver(sub_df, all_df, barcode_target, percentile_th, verbose=0, mi_exclusion_distance=3):
    
    # Create lists of all barcodes and inserts in sub_df (i.e., with similar barcode)
    # and create MSAs separately, then merge with a spacer
    
    barcodes = sub_df['Barcode'].tolist()
    inserts = sub_df['Insert'].tolist()

    mi_matrix, top_cols = calculate_msa_mi_and_top_cols(barcodes, inserts, verbose, mi_exclusion_distance)

    
    